In [0]:
# Replace with your actual catalog, schema, and volume names
volume_path = "/Volumes/dbacademy/default/tutorials/customers/"

# Read all parquet files in the folder (both initial + delta files)
raw_df = spark.read.format("parquet").load(volume_path)

display(raw_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Define window partitioned by primary key, ordered by modified date descending
window_spec = Window.partitionBy("CustomerID").orderBy(col("LastModifiedDate").desc())

# Filter to keep only the latest record (rank = 1)
deduped_df = (
    raw_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Register as a temporary SQL view so we can run SQL MERGE
deduped_df.createOrReplaceTempView("bronze_customers_latest")

display(spark.sql("SELECT * FROM bronze_customers_latest"))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_customers (
    CustomerID INT,
    CustomerName STRING,
    City STRING,
    LastModifiedDate TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- option 1
MERGE INTO
  silver_customers AS target
USING
  bronze_customers_latest AS source
ON
  target.CustomerID = source.CustomerID
WHEN MATCHED AND
  target.LastModifiedDate < source.LastModifiedDate
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- option 2 (not req. here)

-- MERGE INTO silver_customers AS target
-- USING bronze_customers_latest AS source
-- ON target.CustomerID = source.CustomerID
-- WHEN MATCHED THEN
--   UPDATE SET 
--     target.CustomerName = source.CustomerName,
--     target.City = source.City,
--     target.LastModifiedDate = source.LastModifiedDate
-- WHEN NOT MATCHED THEN
--   INSERT (CustomerID, CustomerName, City, LastModifiedDate)
--   VALUES (source.CustomerID, source.CustomerName, source.City, source.LastModifiedDate);

---
### In Delta Lake `MERGE INTO`, when should you use `UPDATE SET *` / `INSERT *` versus explicitly specifying each column mapping?

* Use **`UPDATE SET *` / `INSERT ***` when the source and target schemas are **identical mirror images** and you want to overwrite or insert every single column automatically.
* Use **Explicit Column Mapping (`UPDATE SET target.col = source.col`)** when target and source schemas **differ**, when you need to **preserve audit/history columns**, or when applying **conditional business transformations**.

---

#### Comparison Breakdown

| Feature | `UPDATE SET *` / `INSERT *` | Explicit Mapping (`SET col = ...`) |
| --- | --- | --- |
| **Schema Dependency** | **Strict:** Source and target must have the exact same column names and compatible types; otherwise, Spark throws an analysis error. | **Flexible:** Source can have fewer, more, or differently named columns than the target. |
| **Audit Metadata Columns** | Overwrites everything in the row, including initial creation timestamps. | Allows selective preservation (e.g., updating `modified_date` while leaving `created_date` untouched). |
| **In-Flight Logic** | Direct column-for-column copy; no inline transformations possible. | Allows expressions during merge (e.g., `SET target.City = UPPER(source.City)`). |
| **Maintenance** | Automatically picks up new columns if `WITH SCHEMA EVOLUTION` is enabled. | Requires manual script edits whenever a new column is introduced. |

---

#### When to Use Which in Real Projects

##### 1. When to Use `UPDATE SET *` (Shorthand Syntax)

* **Bronze-to-Silver Replication:** When your Silver table is a clean 1:1 typed copy of the Bronze source table.
* **Tables with 40+ Columns:** Writing out dozens of column mappings manually is tedious and error-prone.
* **Automatic Schema Evolution:** When you want incoming source columns to automatically add themselves into the target Delta table without altering SQL scripts:
```sql
MERGE WITH SCHEMA EVOLUTION INTO silver_customers AS target
USING bronze_customers_latest AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

```

##### 2. When to Use Explicit Column Mapping (Production Standard)

* **Preserving `created_at` or Audit Columns:**
In real pipelines, you want `created_at` to record when the row was first born, and `updated_at` to record when it was modified:
```sql
WHEN MATCHED THEN
  UPDATE SET 
    target.CustomerName = source.CustomerName,
    target.City = source.City,
    target.updated_at = current_timestamp() -- created_at is preserved!
WHEN NOT MATCHED THEN
  INSERT (CustomerID, CustomerName, City, created_at, updated_at)
  VALUES (source.CustomerID, source.CustomerName, source.City, current_timestamp(), current_timestamp());

```


* **Surrogate Keys:** If the target has an auto-incrementing identity or hash key that should never be overwritten by the source.
* **Source Contains Helper Columns:** If your source DataFrame contains intermediate processing columns (like `rn`, `_op_type`, or batch IDs) that do not exist in the final target table. Using `UPDATE SET *` would fail because the target does not have those columns.

In [0]:
%sql
SELECT * FROM silver_customers ORDER BY CustomerID;

In [0]:
%sql
DESCRIBE HISTORY silver_customers;

In [0]:
%sql
-- RESTORE TABLE silver_customers TO VERSION AS OF 1;